# Telecom Customer Churn Analysis

This notebook explores churn patterns, trains and compares classification models, selects a churn-focused probability threshold, reviews model behavior, and exports the model used by the Streamlit app. Run cells from top to bottom.


## 1. Load and inspect the data

The notebook searches for `Customer_Churn.csv` in the project folder and then in the adjacent `DATASET` folder. The CSV is local and is not committed to this repository.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
candidate_paths = [Path("Customer_Churn.csv"), Path.cwd().parent.parent / "DATASET" / "Customer_Churn.csv"]
data_path = next((path for path in candidate_paths if path.is_file()), None)
if data_path is None:
    searched = "\n".join(str(path.resolve()) for path in candidate_paths)
    raise FileNotFoundError(f"Customer_Churn.csv was not found. Place it in the project or adjacent DATASET folder. Checked:\n{searched}")

df = pd.read_csv(data_path)
print(f"Loaded {data_path} with {df.shape[0]:,} rows and {df.shape[1]} columns.")
display(df.head())
df.info()
display(df.isna().sum().sort_values(ascending=False).head(10))

### Section summary

The data is loaded from a portable project-relative location. The initial checks report its dimensions, data types, sample rows, and missing-value counts.


## 2. Clean the data

`TotalCharges` can contain blank strings, making it appear as text. Convert it to numeric, replace blank totals with zero, and validate the target labels before modeling.


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

if "Churn" not in df.columns:
    raise KeyError("The dataset must contain a 'Churn' target column.")
if not set(df["Churn"].dropna().unique()).issubset({"Yes", "No"}):
    raise ValueError("Expected Churn values to be 'Yes' or 'No'.")

print("Missing TotalCharges after cleaning:", int(df["TotalCharges"].isna().sum()))
display(df["TotalCharges"].describe())

### Section summary

`TotalCharges` is numeric with blank entries treated as zero. The target column and its expected `Yes`/`No` values are checked before training.


## 3. Exploratory data analysis

Compare churn prevalence and churn rates across tenure, contract, monthly charges, internet service, and payment method. These are descriptive patterns, not proof of causation.


In [ ]:
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"], fill_value=0)
print(f"Overall churn rate: {df['Churn'].eq('Yes').mean():.1%}")
display(churn_counts.rename("Customer count"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", order=["No", "Yes"], ax=axes[0])
axes[0].set_title("Customer count by churn status")
sns.histplot(data=df, x="tenure", hue="Churn", bins=24, element="step", stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Tenure distribution by churn status")
plt.tight_layout()
plt.show()

display(pd.crosstab(df["Contract"], df["Churn"], normalize="index").round(3))
display(df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]].median().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", order=["No", "Yes"], ax=axes[0])
axes[0].set_title("Monthly charges by churn status")
service_rates = df.assign(ChurnFlag=df["Churn"].eq("Yes").astype(float)).groupby("InternetService", as_index=False)["ChurnFlag"].mean()
sns.barplot(data=service_rates, x="InternetService", y="ChurnFlag", ax=axes[1])
axes[1].set_title("Churn rate by internet service")
axes[1].set_ylabel("Churn rate")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

display(pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index").round(3))

### Section summary

Churn is the minority outcome (about 26.5% in the supplied dataset). The plots and tables show sample differences in tenure, contracts, monthly charges, internet service, and payment method. These patterns help guide modeling but do not explain individual causes.


## 4. Split data and define preprocessing

Remove the identifier and target from the predictors. Stratify the holdout split so train and test retain similar churn proportions. Scale continuous values and one-hot encode categories inside a pipeline to avoid preprocessing leakage.


In [ ]:
X = df.drop(columns=["customerID", "Churn"], errors="ignore")
y = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
missing_numeric = set(numeric_features) - set(X_train.columns)
if missing_numeric:
    raise KeyError(f"Required numeric features are missing: {sorted(missing_numeric)}")
categorical_features = [column for column in X_train.columns if column not in numeric_features]
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Numeric features: {len(numeric_features)}; categorical features: {len(categorical_features)}")

### Section summary

The stratified holdout is kept separate from training. Each model pipeline will learn preprocessing from its training data and safely ignore unseen categories at prediction time.


## 5. Train and compare models

Compare logistic regression, a decision tree, and a random forest on the same holdout set using accuracy, precision, and recall. Cross-validation is run on the training partition only.


In [ ]:
models = {
    "Logistic regression": Pipeline([("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, random_state=42))]),
    "Decision tree": Pipeline([("preprocessor", preprocessor), ("model", DecisionTreeClassifier(random_state=42))]),
    "Random forest": Pipeline([("preprocessor", preprocessor), ("model", RandomForestClassifier(random_state=42))]),
}

comparison_rows = []
for name, candidate in models.items():
    candidate.fit(X_train, y_train)
    prediction = candidate.predict(X_test)
    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
    })
comparison = pd.DataFrame(comparison_rows).set_index("Model").sort_values("Recall", ascending=False)
display(comparison.round(3))

pipe = models["Logistic regression"]
cv_recall = cross_val_score(pipe, X_train, y_train, cv=5, scoring="recall")
print(f"Logistic regression 5-fold recall: {cv_recall.mean():.3f} +/- {cv_recall.std():.3f}")

### Section summary

All three models use the same split and metrics. Logistic regression is selected for probability-threshold tuning and coefficient interpretation; five-fold recall provides a stability check on the training partition.


## 6. Tune the decision threshold

A lower cutoff usually catches more churners while also flagging more customers who would stay. Compare 0.3, 0.4, and 0.5, and use 0.4 for the deployed decision rule.


In [ ]:
y_probability = pipe.predict_proba(X_test)[:, 1]
thresholds = [0.3, 0.4, 0.5]
threshold_rows = []
for threshold in thresholds:
    prediction = (y_probability >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
    })
threshold_results = pd.DataFrame(threshold_rows).set_index("Threshold")
display(threshold_results.round(3))
ax = threshold_results.plot(kind="bar", figsize=(9, 5), ylim=(0, 1), rot=0)
ax.set_title("Held-out performance by churn probability threshold")
ax.set_ylabel("Score")
ax.set_xlabel("Probability threshold")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

CHURN_THRESHOLD = 0.4
y_pred = (y_probability >= CHURN_THRESHOLD).astype(int)
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0))

### Section summary

The 0.4 cutoff prioritizes finding churners over maximizing precision. The table and graph show the measured holdout trade-off; campaign capacity and the cost of missed churners should guide any future threshold change.


## 7. Review errors and model coefficients

Inspect the confusion matrix, false negatives, and logistic-regression coefficients. Coefficients describe associations learned by the model and should not be treated as causal effects.


In [ ]:
matrix = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted no churn", "Predicted churn"],
            yticklabels=["Actual no churn", "Actual churn"], ax=ax)
ax.set_title(f"Confusion matrix at threshold {CHURN_THRESHOLD:.1f}")
plt.tight_layout()
plt.show()

feature_names = pipe.named_steps["preprocessor"].get_feature_names_out()
coefficients = pipe.named_steps["model"].coef_[0]
coefficient_table = pd.DataFrame({"Feature": feature_names, "Coefficient": coefficients})
print("Features most associated with a higher churn score:")
display(coefficient_table.nlargest(10, "Coefficient").reset_index(drop=True).round(3))
print("Features most associated with a lower churn score:")
display(coefficient_table.nsmallest(10, "Coefficient").reset_index(drop=True).round(3))

false_negatives = X_test.loc[(y_test == 1) & (y_pred == 0)]
print(f"False negatives at threshold {CHURN_THRESHOLD:.1f}: {len(false_negatives)}")
display(false_negatives[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2))

### Section summary

The confusion matrix distinguishes false positives from missed churners. The coefficient summaries describe the fitted logistic model, while the false-negative profile helps identify customer groups the selected threshold misses.


## 8. Explain predictions with LIME and SHAP

LIME explains one selected customer by making small feature changes and observing how the model score responds. SHAP assigns additive contributions to features; the summary plot compares their influence across test customers, and the waterfall plot explains one customer. Both methods use the same transformed features the model receives.


In [ ]:
# Apply the fitted pipeline's preprocessing once so explainers use the model's exact feature space.
explain_preprocessor = pipe.named_steps["preprocessor"]
explain_model = pipe.named_steps["model"]
explain_feature_names = explain_preprocessor.get_feature_names_out()

# One-hot encoding often produces a sparse matrix. Convert it only when needed.
def make_dense(feature_matrix):
    if hasattr(feature_matrix, "toarray"):
        return feature_matrix.toarray()
    return np.asarray(feature_matrix)

X_train_explain = make_dense(explain_preprocessor.transform(X_train))
X_test_explain = make_dense(explain_preprocessor.transform(X_test))

lime_explainer = LimeTabularExplainer(
    training_data=X_train_explain,
    feature_names=list(explain_feature_names),
    class_names=["No churn", "Churn"],
    mode="classification",
    discretize_continuous=True,
    random_state=42,
)

# Explain the first held-out customer; the classifier receives transformed values.
lime_explanation = lime_explainer.explain_instance(
    X_test_explain[0],
    explain_model.predict_proba,
    num_features=min(12, len(explain_feature_names)),
)
display(pd.DataFrame(lime_explanation.as_list(), columns=["Feature condition", "LIME contribution"]))
lime_figure = lime_explanation.as_pyplot_figure()
lime_figure.tight_layout()
plt.show()

In [ ]:
# Use a small training background and a test subset to keep SHAP plots responsive.
shap_background = pd.DataFrame(
    X_train_explain[:min(100, len(X_train_explain))], columns=explain_feature_names
)
shap_test = pd.DataFrame(
    X_test_explain[:min(200, len(X_test_explain))], columns=explain_feature_names
)
shap_explainer = shap.LinearExplainer(explain_model, shap_background)
shap_values = shap_explainer(shap_test)

# Each point is a feature contribution for a test customer; larger absolute values have greater impact.
shap.plots.beeswarm(shap_values, max_display=15)

# Explain the first test customer's prediction by showing each feature's contribution.
shap.plots.waterfall(shap_values[0], max_display=15)

# Show how tenure values relate to their SHAP contribution across explained customers.
shap.plots.scatter(shap_values[:, "numeric__tenure"])

### Section summary

LIME gives a local, model-agnostic explanation for one customer. SHAP gives additive feature contributions, with a beeswarm view across the test sample and a waterfall view for one customer. Both operate on the pipeline's transformed inputs, so the encoded feature names line up with the model. Explanations describe model behavior; they do not establish causality.


## 9. Export the model for Streamlit

Save the fitted preprocessing-and-logistic-regression pipeline. The app loads this file and applies the same 0.4 cutoff.


In [ ]:
model_path = Path("churn_model.pkl")
joblib.dump(pipe, model_path)
print(f"Saved fitted model to {model_path.resolve()}")

### Section summary

The exported artifact includes preprocessing and classification. Re-run the notebook after changing training data or model settings so the saved artifact remains in sync with this analysis and the Streamlit app.


## 10. Final results and project summary

The selected model is logistic regression at the recall-focused 0.4 threshold. The following results are computed from the stratified holdout when the notebook runs.


In [ ]:
matrix = confusion_matrix(y_test, y_pred)
final_results = pd.Series({
    "Selected model": "Logistic regression pipeline",
    "Decision threshold": CHURN_THRESHOLD,
    "Holdout accuracy": accuracy_score(y_test, y_pred),
    "Holdout precision": precision_score(y_test, y_pred, zero_division=0),
    "Holdout recall": recall_score(y_test, y_pred, zero_division=0),
    "True positives": int(matrix[1, 1]),
    "False negatives": int(matrix[1, 0]),
    "False positives": int(matrix[0, 1]),
})
display(final_results.to_frame("Result"))

### Recorded holdout results

| Metric | Result |
|---|---:|
| Accuracy | 77.7% |
| Precision | 56.8% |
| Recall | 66.8% |
| True positives | 250 |
| False negatives | 124 |
| False positives | 190 |

The 5-fold training-partition recall averaged 54.8% (standard deviation 2.4%). The 0.4 cutoff increases recall compared with the 0.5 default while also increasing false positives. The results cell above recomputes these metrics from the current data each time the notebook is run.


### Section summary

Loaded and inspected the telecom dataset; cleaned total charges; explored churn patterns; built a leakage-safe preprocessing pipeline; compared three classifiers; selected and evaluated a 0.4 threshold; reviewed model errors and coefficients; explained predictions with LIME and SHAP; and saved the model for Streamlit deployment.
